# 10 个问题的谜题 ― 用 Implies 解开自指问答谜题

## 本章要点

我们来解一个 10 个问题**互相参照对方答案**的谜题。

- 把每个问题的答案表示成 1〜5 的整数变量
- 用 `Implies()` 写“如果答案是某某，则这个条件成立”
- 全探索要 5¹⁰ ≈ 976 万种的问题，直接写成约束

本章的中心就是 `Implies()`（蕴含）的用法。

## 问题

```
1. 第一个答案是“b”的问题是哪个？
（a）2；（b）3；（c）4；（d）5；（e）6

2. 唯一一组连续两题答案相同的是？
（a）2，3；（b）3，4；（c）4，5；（d）5，6；（e）6，7；

3. 与本题答案相同的问题是哪个？
（a）1；（b）2；（c）4；（d）7；（e）6

4. 答案是“a”的问题有几个？
（a）0；（b）1；（c）2；（d）3；（e）4

5. 与本题答案相同的问题是哪个？
（a）10；（b）9；（c）8；（d）7；（e）6

6. 答案是“a”的问题个数，与答案是哪个选项的问题个数相同？
（a）b；（b）c；（c）d；（d）e；（e）以上都不是

7. 按字母顺序，本题答案与下一题答案相差几个字母？
（a）4；（b）3；（c）2；（d）1；（e）0。（※a 与 b 相差 1 个字母）

8. 答案是元音字母（a, e）的问题有几个？
（a）2；（b）3；（c）4；（d）5；（e）6。（※元音字母是 a 和 e）

9. 答案是辅音字母（b, c, d）的问题个数是？
（a）质数；（b）阶乘数；（c）平方数；（d）立方数；（e）5 的倍数

10. 本题的答案是？
（a）a；（b）b；（c）c；（d）d；（e）e
```

## 难点在哪里

- **条件互相依赖**：Q3 的答案必须与另一题的答案一致，Q6 的条件则是比较“答案为 a 的个数”和“答案为其他选项的个数”。
- **组合很多**：1 题 5 个选项、共 10 题，老老实实全探索有 $5^{10} = 9{,}765{,}625$ 种。
- **有些条件不是普通等式**：Q7 是答案之差的绝对值，Q9 是辅音的个数属于质数、平方数、阶乘数、5 的倍数之一。

人手已经无能为力，所以把条件原样交给 Z3。

## 思路

把问题 $i$ 的答案表示成整数变量 `q[i]`。取值 1〜5，1=a、2=b、3=c、4=d、5=e。

每个问题都是“**如果 Q_i 的答案是某某，则这个条件成立**”的形式。这正是 `Implies()`。比如 Q4 可以写成

```python
solver.add([Implies(q[4] == ans, count_a() == value) for ans, value in zip(choices, [0, 1, 2, 3, 4])])
```

“Q4 的答案是 a(1)，则 a 的个数是 0”“是 b(2)，则是 1”……一次性全部加进去。

Q10“本题的答案是？(a)a (b)b …”选哪个都不矛盾（因为它只是指向自己）。所以不加约束。

In [1]:
from z3 import *

# 选项 a〜e 用整数 1〜5 表示（不是 Z3 变量，只是编号）
a, b, c, d, e = range(1, 6)
choices = [a, b, c, d, e]

# 10 个问题的答案
q = {i: Int(f'q{i}') for i in range(1, 11)}
solver = Solver()

# 每个答案都在 1〜5（a〜e 之一）
for x in q.values():
    solver.add(1 <= x, x <= 5)


def first_b(idx):
    """“第一个答案是 b 的是问题 idx”这个条件"""
    return And([q[i] != b for i in range(1, idx)] + [q[idx] == b])


def only_equals(idx):
    """“相邻且答案相同的只有 idx 和 idx+1 这一组”这个条件"""
    return And([q[i] == q[i + 1] if i == idx else q[i] != q[i + 1]
                for i in range(1, 10)])


def count_answer(ans):
    """答案为 ans 的问题个数"""
    return Sum([x == ans for x in q.values()])


def is_in(expr, values):
    """expr 等于 values 中的某一个"""
    return Or([expr == v for v in values])


# Q1: 第一个答案是 b 的问题
solver.add([Implies(q[1] == ans, first_b(idx)) for ans, idx in zip(choices, [2, 3, 4, 5, 6])])

# Q2: 唯一一组相邻且答案相同
solver.add([Implies(q[2] == ans, only_equals(idx)) for ans, idx in zip(choices, [2, 3, 4, 5, 6])])

# Q3: 与本题答案相同的问题
solver.add([Implies(q[3] == ans, q[3] == q[idx]) for ans, idx in zip(choices, [1, 2, 4, 7, 6])])

# Q4: a 的个数
solver.add([Implies(q[4] == ans, count_answer(a) == value) for ans, value in zip(choices, [0, 1, 2, 3, 4])])

# Q5: 与本题答案相同的问题
solver.add([Implies(q[5] == ans, q[5] == q[idx]) for ans, idx in zip(choices, [10, 9, 8, 7, 6])])

# Q6: a 的个数与哪个答案的个数相同
solver.add(
    Implies(q[6] == a, count_answer(a) == count_answer(b)),
    Implies(q[6] == b, count_answer(a) == count_answer(c)),
    Implies(q[6] == c, count_answer(a) == count_answer(d)),
    Implies(q[6] == d, count_answer(a) == count_answer(e)),
    Implies(q[6] == e, And(count_answer(a) != count_answer(b),
                           count_answer(a) != count_answer(c),
                           count_answer(a) != count_answer(d),
                           count_answer(a) != count_answer(e))),
)

# Q7: 本题与下一题的答案，按字母顺序相差几个字母
solver.add([Implies(q[7] == ans, Abs(q[7] - q[8]) == diff) for ans, diff in zip(choices, [4, 3, 2, 1, 0])])

# Q8: 元音（a, e）的个数
solver.add([Implies(q[8] == ans, count_answer(a) + count_answer(e) == value)
            for ans, value in zip(choices, [2, 3, 4, 5, 6])])

# Q9: 辅音（b, c, d）的个数是质数、阶乘数、平方数、立方数、5 的倍数之一
consonants = count_answer(b) + count_answer(c) + count_answer(d)
solver.add([Implies(q[9] == ans, is_in(consonants, values)) for ans, values in
            zip(choices, [[2, 3, 5, 7],      # 质数
                          [1, 2, 6],         # 阶乘数 (1!, 2!, 3!)
                          [1, 4, 9],         # 平方数
                          [1, 8],            # 立方数
                          [5, 10]])])        # 5 的倍数

# Q10 是自指的，所以不加约束

if solver.check() == sat:
    m = solver.model()
    answers = [m[q[i]].as_long() for i in range(1, 11)]
    print("答案:", "".join("abcde"[v - 1] for v in answers))
    for i, v in enumerate(answers, 1):
        print(f"Q{i} = {'abcde'[v - 1]}")
else:
    print("无解")

答案: cdebeedcba
Q1 = c
Q2 = d
Q3 = e
Q4 = b
Q5 = e
Q6 = e
Q7 = d
Q8 = c
Q9 = b
Q10 = a


答案只有 `cdebeedcba` 这一种。

| Q1 | Q2 | Q3 | Q4 | Q5 | Q6 | Q7 | Q8 | Q9 | Q10 |
|---|---|---|---|---|---|---|---|---|---|
| c | d | e | b | e | e | d | c | b | a |

Q9 的“5 的倍数”取 5 和 10（1 不是 5 的倍数，所以没有放进去）。

解只有一个，这一点可以这样确认：把找到的答案作为禁止条件加进去，再 `check()` 一次（如果返回 `unsat`，就说明没有别的答案）。

## 什么是 Implies

`Implies(P, Q)` 是**“P 为真则 Q 也为真”**的条件。在 Z3 里，它和

$$
\text{Implies}(P, Q) \equiv \neg P \vee Q
$$

含义相同。

| $P$ | $Q$ | $\text{Implies}(P, Q)$ |
|:---:|:---:|:---:|
| False | False | True |
| False | True | True |
| True | False | **False** |
| True | True | True |

- `P` 为假时，不管 `Q` 是什么，整体都为真（这是逻辑的规矩：假可以推出任何东西）。
- 只有 `P` 为真而 `Q` 为假时，整体才为假。

有这条性质，就能写出“**只在某个条件成立时**才适用另一条条件”的约束。

用一个小例子确认一下。`x` 和 `y` 取 1〜4，加上下面两条约束。

- `Implies(x >= 3, y <= 2)` … x 是 3 或 4 时，y 是 1 或 2
- `Implies(x <= 2, y >= 3)` … x 是 1 或 2 时，y 是 3 或 4

| `x` | `y` 可能的值 |
|---|---|
| 1, 2 | 3, 4 |
| 3, 4 | 1, 2 |

组合共有 $2 \times 2 \times 2 = 8$ 种。

In [2]:
x, y = Ints('x y')
s = Solver()
s.add(1 <= x, x <= 4)
s.add(1 <= y, y <= 4)
s.add(Implies(x >= 3, y <= 2))
s.add(Implies(x <= 2, y >= 3))

while s.check() == sat:
    m = s.model()
    print(m)
    s.add(Or([d() != m[d] for d in m]))   # 禁止已找到的解，继续找下一个解

[y = 1, x = 3]
[y = 2, x = 4]
[x = 2, y = 3]
[x = 2, y = 4]
[x = 1, y = 3]
[y = 2, x = 3]
[x = 4, y = 1]
[x = 1, y = 4]


8 种解全都出来了。可以看出 `Implies` 确实在起“条件约束”的作用。

另外，像这个谜题这样**每个选项对应不同条件**的问题，把 `Implies` 按选项个数摆开是标准做法。用 `zip(choices, [...])` 可以一行就把选项和条件的对应写出来。

## 小结

- 自指谜题只要“把每个问题的答案变成变量”，就变成了普通的约束问题
- “如果答案是某某……”用 `Implies()` 写。有几个选项就摆几条
- 计数（`Sum([x == ans for x in q.values()])`）、绝对值（`Abs`）、“属于其中某一个”（`Or`）也都能当约束用
- 全探索要 976 万种的问题，写成约束后 Z3 一瞬间就解出来了